# Comparison of performance of legacy DeLUCS code with updated DeLUCS code

This notebook serves to compare the performance of the legacy DeLUCS pipeline with the recent patch (using the newly introduced features to mimic the behaviour of the DeLUCS pipeline in the most recent commit prior to this patch: 70e5ce7ef199a36a90dc961b959a23e180289d67 on August 31, 2022). This patch includes various feature additions, as well as fixes. Performance of the updated pipeline is expected to be lower on average, on account of fixing a label leakage issue. However, performance may still be higher on single tests, due to variance in the performance of the pipeline.

This patch provides the necessary features to recreate the results of the legacy DeLUCS pipeline, with the exception of fixing an off-by-one error in k-mer counting, where a legacy feature was not provided.

To recreate the legacy behaviour of the DeLUCS pipeline during evaluation, a user should:
1. Run get_pairs.py with k=6, and with 'modify' set to 'duplication' (n\_mimics should be set to 3 for all of the eleven datasets, except for Actinopterygii, Ostariophysi, Cypriniformes, and Cyprinidae, where it should instead be set to 8)\*.
2. Run EvaluateDeLUCS_legacy_alignment.py, with k=6.

\* Technically, to recreate get_pairs.py according to legacy behaviour, those four specified datasets should be run with n\_mimics set to 18 (since 3(n-2) were being generated). We use n_\mimics set to 8 to match the pair counts between the two configurations.

Installed library versions for this experiment:

In [1]:
import numpy, torch, cython, matplotlib, cv2, sklearn, scipy, sys
print("Python version: " + str(sys.version))
print("Numpy version: " + str(numpy.__version__))
print("Torch version: " + str(torch.__version__))
print("Cython version: " + str(cython.__version__))
print("Matplotlib version: " + str(matplotlib.__version__))
print("OpenCV version: " + str(cv2.__version__))
print("Scikit-learn version: " + str(sklearn.__version__))
print("Scipy version: " + str(scipy.__version__))

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Numpy version: 2.4.6
Torch version: 2.14.1+cu130
Cython version: 3.0.12
Matplotlib version: 3.10.0
OpenCV version: 4.14.0
Scikit-learn version: 1.6.1
Scipy version: 1.16.3


## Initial setup

First, we mount the Drive where we have the files.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/My Drive/DeLUCS/src

/content/drive/My Drive/DeLUCS/src


We then install the BioPython Dependencies.

In [4]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.4 MB/s  0:00:00


We now build the datasets.

In [5]:
!python build_dp.py --data_path='../data/Vertebrata/Test Files' --output_dir='../data/sequence datasets' --output_file='vertebrata.p'
!python build_dp.py --data_path='../data/Fish/Test Files/Actinopterygii' --output_dir='../data/sequence datasets' --output_file='actinopterygii.p'
!python build_dp.py --data_path='../data/Fish/Test Files/Neopterygii' --output_dir='../data/sequence datasets' --output_file='neopterygii.p'
!python build_dp.py --data_path='../data/Fish/Test Files/Ostariophysi' --output_dir='../data/sequence datasets' --output_file='ostariophysi.p'
!python build_dp.py --data_path='../data/Fish/Test Files/Cypriniformes' --output_dir='../data/sequence datasets' --output_file='cypriniformes.p'
!python build_dp.py --data_path='../data/Fish/Test Files/Cyprinidae' --output_dir='../data/sequence datasets' --output_file='cyprinidae.p'
!python build_dp.py --data_path='../data/Bacteria/Test_Files' --output_dir='../data/sequence datasets' --output_file='bacteria.p'
!python build_dp.py --data_path='../data/Bacteria/Proteo_Test_Files' --output_dir='../data/sequence datasets' --output_file='proteobacteria.p'
!python build_dp.py --data_path='../data/Influenza-A/Test Files' --output_dir='../data/sequence datasets' --output_file='influenza.p'
!python build_dp.py --data_path='../data/Dengue/Test Files' --output_dir='../data/sequence datasets' --output_file='dengue.p'
!python build_dp.py --data_path='../data/HBV/Test Files' --output_dir='../data/sequence datasets' --output_file='hbv.p'

Streaming output truncated to the last 5000 lines.
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900102485.1.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900101105.1_2.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900101105.1_5.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900102485.1_3.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900101105.1_4.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900102485.1_2.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_006539305.1_7.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900116045.1_2.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900129975.1_5.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900116045.1.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900116045.1_3.fas
../data/Bacteria/Proteo_Test_Files/Desulfovibrionaceae/GCF_900129975.1.fas
../data/Bacteria/Proteo_Test_Fi

## Computing k-mer count pairs

We compute the k-mer count pairs for the sequences, starting with the sequence/mimic pairs.

In [10]:
!python get_pairs.py --data_path='../data/sequence datasets/vertebrata.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/vertebrata_mutation_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/actinopterygii.p' --k=6 --n_mimics=8 --modify='mutation' --output='../data/kmer pairs datasets/actinopterygii_mutation_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/neopterygii.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/neopterygii_mutation_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/ostariophysi.p' --k=6 --n_mimics=8 --modify='mutation' --output='../data/kmer pairs datasets/ostariophysi_mutation_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/cypriniformes.p' --k=6 --n_mimics=8 --modify='mutation' --output='../data/kmer pairs datasets/cypriniformes_mutation_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/cyprinidae.p' --k=6 --n_mimics=8 --modify='mutation' --output='../data/kmer pairs datasets/cyprinidae_mutation_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/bacteria.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/bacteria_mutation_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/proteobacteria.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/proteobacteria_mutation_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/influenza.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/influenza_mutation_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/dengue.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/dengue_mutation_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/hbv.p' --k=6 --n_mimics=3 --modify='mutation' --output='../data/kmer pairs datasets/hbv_mutation_mimics_k6_n3.p' --seed=0

............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....
............computing learning pairs................
......saving mutated pairs.....


We also create a dataset of sequence/sequence pairs, to match the behaviour of the legacy DeLUCS pipeline.

In [11]:
!python get_pairs.py --data_path='../data/sequence datasets/vertebrata.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/vertebrata_duplication_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/actinopterygii.p' --k=6 --n_mimics=8 --modify='duplication' --output='../data/kmer pairs datasets/actinopterygii_duplication_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/neopterygii.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/neopterygii_duplication_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/ostariophysi.p' --k=6 --n_mimics=8 --modify='duplication' --output='../data/kmer pairs datasets/ostariophysi_duplication_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/cypriniformes.p' --k=6 --n_mimics=8 --modify='duplication' --output='../data/kmer pairs datasets/cypriniformes_duplication_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/cyprinidae.p' --k=6 --n_mimics=8 --modify='duplication' --output='../data/kmer pairs datasets/cyprinidae_duplication_mimics_k6_n8.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/bacteria.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/bacteria_duplication_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/proteobacteria.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/proteobacteria_duplication_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/influenza.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/influenza_duplication_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/dengue.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/dengue_duplication_mimics_k6_n3.p' --seed=0
!python get_pairs.py --data_path='../data/sequence datasets/hbv.p' --k=6 --n_mimics=3 --modify='duplication' --output='../data/kmer pairs datasets/hbv_duplication_mimics_k6_n3.p' --seed=0

Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....
Computing training pairs
.....saving data.....


## Pipeline evaluation

We now run the training and evaluation of the patched DeLUCS.

In [12]:
print("\n\nVertebrata:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='vertebrata_mutation_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nActinopterygii:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='actinopterygii_mutation_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nNeopterygii:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='neopterygii_mutation_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nOstariophysi:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='ostariophysi_mutation_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nCypriniformes:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='cypriniformes_mutation_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nCyprinidae:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='cyprinidae_mutation_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nBacteria:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='bacteria_mutation_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nProteobacteria:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='proteobacteria_mutation_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nInfluenza:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='influenza_mutation_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nDengue:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='dengue_mutation_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nHBV:")
!python EvaluateDeLUCS.py --data_dir='../data/kmer pairs datasets' --input_file='hbv_mutation_mimics_k6_n3.p' --seed=0 --k=6



Vertebrata:
The size of the training array is: (7500, 2, 4096)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worke

We also run the training and evaluation of the legacy DeLUCS, where it uses the ground truth labels to inform the ensembling process. We expect, on average, the results to be better in the legacy version, as a result of its supervised nature.

In [13]:
print("\n\nVertebrata:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='vertebrata_duplication_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nActinopterygii:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='actinopterygii_duplication_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nNeopterygii:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='neopterygii_duplication_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nOstariophysi:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='ostariophysi_duplication_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nCypriniformes:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='cypriniformes_duplication_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nCyprinidae:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='cyprinidae_duplication_mimics_k6_n8.p' --seed=0 --k=6
print("\n\nBacteria:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='bacteria_duplication_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nProteobacteria:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='proteobacteria_duplication_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nInfluenza:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='influenza_duplication_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nDengue:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='dengue_duplication_mimics_k6_n3.p' --seed=0 --k=6
print("\n\nHBV:")
!python EvaluateDeLUCS_legacy_alignment.py --data_dir='../data/kmer pairs datasets' --input_file='hbv_duplication_mimics_k6_n3.p' --seed=0 --k=6



Vertebrata:
The size of the training array is: (7500, 2, 4096)
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worke